# Directed Speech Detector: full-scale GPU run

**Status: NOT YET RUN.** This notebook has not been executed; every number in the repository README comes from the CPU-scale runs described in `docs/EXPERIMENTS.md`.
It scales the same code up (60k synthetic-proxy training utterances, a larger model) and shows how to plug in real audio via a CSV manifest.

Remember: the default data is a **synthetic proxy**, not real assistant data (see `docs/DATA.md`).

Runtime: *Runtime > Change runtime type > GPU*.

In [ ]:
!git clone https://github.com/litheesh-v-rambabu/directed-speech-detector.git
%cd directed-speech-detector
# Colab already ships a CUDA build of torch, so pip only adds the lightweight extras.
!pip install -q -e ".[serve,export,viz]"


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Single full-scale run (audio + text + speaker)
Data generation uses CPU workers and is cached under `data/cache`; training uses the GPU (`train.device: auto`).

In [ ]:
!dsd train --config configs/colab_full.yaml

## 2. Full-scale ablations (3 seeds)
Same four ablations as the CPU study, with the larger model and dataset. Roughly 3 x 4 trainings; reduce `--seeds` to save time.

In [ ]:
OVERRIDES = "data.n_train=60000 data.n_val=5000 data.n_test=10000 data.cache_dir=data/cache_full data.num_workers=2 " \
            "model.d_model=192 model.n_heads=6 model.audio_layers=6 model.text_layers=4 " \
            "train.batch_size=128 train.epochs=30 train.patience=8 output_dir=runs_full"
!dsd ablate --seeds 0 1 2 --set $OVERRIDES
!dsd report --runs-dir runs_full --docs-dir docs_full

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open("docs_full/results/ablation.md").read()))
display(Image("docs_full/figures/ablation.png"))

## 3. Export and benchmark (CPU-side measurement)
Latency numbers are only comparable to the README if you also state the Colab CPU model; the benchmark records `hardware` in its JSON.

In [ ]:
!dsd export --run-dir runs_full/fused_speaker_s0
!dsd benchmark --run-dir runs_full/fused_speaker_s0 --docs-dir docs_full
print(open("docs_full/results/benchmark.md").read())

## 4. Real audio via a manifest (optional, untested)

The pipeline accepts real 16 kHz mono 16-bit WAVs through `data/real/{train,val,test}.csv` with columns `audio_path,transcript,label`
(`1` = directed at the assistant, `0` = background). Real corpora have no enrolled-speaker profile, so use the audio + text configuration (`configs/real_audio_template.yaml`).

The cell below sketches a LibriSpeech-based *background* source. Directed examples (e.g. SLURP / Fluent Speech Commands) must be downloaded and
converted to the same manifest format by hand. **This cell has never been executed**; check licences (LibriSpeech: CC BY 4.0) and adapt paths before use.

In [ ]:
# UNTESTED sketch: write LibriSpeech dev-clean as label-0 rows (needs torchaudio; resample to 16 kHz mono).
import csv, os, wave
import numpy as np, torchaudio
os.makedirs("data/real/wavs", exist_ok=True)
ds = torchaudio.datasets.LIBRISPEECH("data/librispeech", url="dev-clean", download=True)
rows = []
for i in range(2000):
    waveform, sr, text, *_ = ds[i]
    assert sr == 16_000
    pcm = (waveform[0].clamp(-1, 1).numpy() * 32767).astype("<i2")
    path = f"wavs/ls_{i}.wav"
    with wave.open(f"data/real/{path}", "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(16_000); w.writeframes(pcm.tobytes())
    rows.append((path, text.lower(), 0))
with open("data/real/background_rows.csv", "w", newline="") as f:
    csv.writer(f).writerows([("audio_path", "transcript", "label"), *rows])

In [ ]:
!dsd train --config configs/real_audio_template.yaml  # after train/val/test.csv exist

## 5. Save results
Copy `runs_full/`, `docs_full/` to Drive, then record the measured numbers in `docs/EXPERIMENTS.md` (command, config, hardware) before quoting them anywhere.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp -r runs_full docs_full /content/drive/MyDrive/directed_speech_detector_full/